# MuseTalk 1.5 GPU validation worker

Clean Colab T4 worker. Uses FP16 only for the GPU inference models, keeps CPU-side/auxiliary models FP32, and validates CUDA before starting so it cannot silently spend 15+ minutes in CPU mode.

In [ ]:
# 1. Runtime and repository
!nvidia-smi
!pip -q install uv
import subprocess
from pathlib import Path
MT=Path('/content/MuseTalk'); VENV=Path('/content/musetalk310')
if not MT.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/TMElyralab/MuseTalk.git',str(MT)],check=True)
if not (VENV/'bin/python').exists(): subprocess.run(['uv','venv','--python','3.10',str(VENV)],check=True)
PY=str(VENV/'bin/python'); subprocess.run([PY,'-V'],check=True)
probe=subprocess.run([PY,'-c','import torch; print("torch=",torch.__version__); print("cuda_available=",torch.cuda.is_available()); print("cuda=",torch.version.cuda); print("gpu=",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")'],text=True,capture_output=True,check=True)
print(probe.stdout)
if 'cuda_available= True' not in probe.stdout:
    raise RuntimeError('CUDA/T4 is NOT available inside the MuseTalk Python environment. Stop here; do not run MuseTalk on CPU. In Colab choose Runtime > Change runtime type > T4 GPU, then rerun from the top.')
print('T4 CUDA runtime is ready.')


In [ ]:
# 2. Dependencies and required model weights
import subprocess
from pathlib import Path
PY='/content/musetalk310/bin/python'; MT='/content/MuseTalk'
subprocess.run(['uv','pip','install','--python',PY,'pip','setuptools','wheel'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'torch==2.0.1','torchvision==0.15.2','torchaudio==2.0.2','--index-url','https://download.pytorch.org/whl/cu118'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'-r',MT+'/requirements.txt'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'openmim','gdown'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'--no-build-isolation','chumpy==0.70'],check=True)
MIM='/content/musetalk310/bin/mim'
for pkg in ['mmengine','mmcv==2.0.1','mmdet==3.1.0','mmpose==1.1.0']: subprocess.run([MIM,'install',pkg],check=True)
# Force the official MuseTalk VAE .bin weights; diffusers must not search for safetensors.
vae_py=Path(MT)/'musetalk/models/vae.py'
vae_text=vae_py.read_text()
old='AutoencoderKL.from_pretrained(self.model_path)'
new='AutoencoderKL.from_pretrained(self.model_path, use_safetensors=False)'
if old in vae_text: vae_py.write_text(vae_text.replace(old,new))
MODELS=Path('/content/MuseTalk/models'); MODELS.mkdir(parents=True,exist_ok=True)
def fetch(url,target,min_bytes):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>=min_bytes: print('ready:',target); return
    print('Downloading:',target)
    subprocess.run(['curl','-L','--fail','--retry','8','--retry-delay','5','--retry-connrefused','-C','-','-o',str(target),url],check=True)
    if target.stat().st_size<min_bytes: raise RuntimeError(f'Incomplete download: {target}')
fetch('https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/unet.pth?download=true',MODELS/'musetalkV15/unet.pth',3000000000)
fetch('https://huggingface.co/TMElyralab/MuseTalk/resolve/main/musetalkV15/musetalk.json?download=true',MODELS/'musetalkV15/musetalk.json',500)
fetch('https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/config.json?download=true',MODELS/'sd-vae/config.json',500)
fetch('https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.bin?download=true',MODELS/'sd-vae/diffusion_pytorch_model.bin',300000000)
fetch('https://huggingface.co/openai/whisper-tiny/resolve/main/config.json?download=true',MODELS/'whisper/config.json',500)
fetch('https://huggingface.co/openai/whisper-tiny/resolve/main/preprocessor_config.json?download=true',MODELS/'whisper/preprocessor_config.json',1000)
fetch('https://huggingface.co/openai/whisper-tiny/resolve/main/pytorch_model.bin?download=true',MODELS/'whisper/pytorch_model.bin',100000000)
fetch('https://huggingface.co/yzd-v/DWPose/resolve/main/dw-ll_ucoco_384.pth?download=true',MODELS/'dwpose/dw-ll_ucoco_384.pth',300000000)
fetch('https://huggingface.co/ManyOtherFunctions/face-parse-bisent/resolve/main/79999_iter.pth?download=true',MODELS/'face-parse-bisent/79999_iter.pth',50000000)
fetch('https://huggingface.co/ManyOtherFunctions/face-parse-bisent/resolve/main/resnet18-5c106cde.pth?download=true',MODELS/'face-parse-bisent/resnet18-5c106cde.pth',40000000)
print('All dependencies and weights are ready.')


In [ ]:
# 3. Upload inputs and run MuseTalk
from google.colab import files
from pathlib import Path
import subprocess, os
print('Upload the APPROVED singer image/video:')
avatar=next(iter(files.upload()))
print('Upload the successful ACE-Step bhajan MP3/WAV:')
audio=next(iter(files.upload()))
assert Path(avatar).suffix.lower() in {'.png','.jpg','.jpeg','.webp','.mp4','.mov','.webm'}
assert Path(audio).suffix.lower() in {'.mp3','.wav','.m4a','.flac','.aac','.ogg'}
OUT=Path('/content/musetalk_output'); OUT.mkdir(parents=True,exist_ok=True)
avatar_src=OUT/'avatar_source.png'; audio_wav=OUT/'audio.wav'
subprocess.run(['ffmpeg','-y','-i',avatar,'-frames:v','1','-vf','scale=512:-2','-pix_fmt','rgb24',str(avatar_src)],check=True)
subprocess.run(['ffmpeg','-y','-i',audio,'-ar','16000','-ac','1',str(audio_wav)],check=True)
cfg=Path('/content/MuseTalk/configs/inference/test.yaml')
cfg.write_text(f'''bhajan_test:
  video_path: "{avatar_src}"
  audio_path: "{audio_wav}"
  result_name: "bhajan_lipsync.mp4"
''')
os.chdir('/content/MuseTalk')
env=os.environ.copy(); env['MPLBACKEND']='Agg'; env['PYTHONPATH']='/content/MuseTalk:'+env.get('PYTHONPATH',''); env['CUDA_VISIBLE_DEVICES']='0'
gpu=subprocess.run(['/content/musetalk310/bin/python','-c','import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")'],text=True,capture_output=True,check=True)
print('GPU check:',gpu.stdout.strip())
if not gpu.stdout.startswith('True'):
    raise RuntimeError('MuseTalk Python cannot see the T4. Refusing to run CPU inference.')
result_dir=Path('/content/musetalk_output/result'); result_dir.mkdir(parents=True,exist_ok=True)
# IMPORTANT: --use_float16 applies FP16 to the GPU VAE/UNet/Whisper path.
# Face parsing and landmark models remain FP32; they are not converted to Half.
# This is the correct fast T4 configuration; do NOT use the old global float32 mode.
cmd=['/content/musetalk310/bin/python','-m','scripts.inference','--inference_config','configs/inference/test.yaml','--result_dir',str(result_dir),'--unet_model_path','models/musetalkV15/unet.pth','--unet_config','models/musetalkV15/musetalk.json','--whisper_dir','models/whisper','--version','v15','--fps','25','--batch_size','2','--use_float16','--parsing_mode','jaw']
print('Starting MuseTalk 1.5 on T4 (GPU FP16 + auxiliary FP32)...')
r=subprocess.run(cmd,env=env,text=True,capture_output=True)
print(r.stdout)
if r.stderr: print(r.stderr)
candidates=list(result_dir.rglob('*.mp4'))
if not candidates:
    frame_dirs=[p for p in result_dir.rglob('*') if p.is_dir() and list(p.glob('*.png'))]
    if not frame_dirs:
        raise RuntimeError('MuseTalk failed before producing frames. See the MuseTalk STDERR above.')
    frame_dir=max(frame_dirs,key=lambda p:len(list(p.glob('*.png')))); frames=sorted(frame_dir.glob('*.png'))
    temp=OUT/'recovered_video.mp4'; candidate=OUT/'bhajan_lipsync.mp4'
    print('MuseTalk made',len(frames),'frames; recovering MP4 with ffmpeg...')
    subprocess.run(['ffmpeg','-y','-v','error','-framerate','25','-i',str(frame_dir/'%08d.png'),'-c:v','libx264','-pix_fmt','yuv420p','-crf','18',str(temp)],check=True)
    subprocess.run(['ffmpeg','-y','-v','error','-i',str(temp),'-i',str(audio_wav),'-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-shortest',str(candidate)],check=True)
else:
    candidate=max(candidates,key=lambda p:p.stat().st_size)
if r.returncode != 0 and (not candidate.exists() or candidate.stat().st_size<=100000):
    raise RuntimeError(f'MuseTalk failed with exit code {r.returncode}. See STDERR above.')
assert candidate.exists() and candidate.stat().st_size>100000
print('SUCCESS:',candidate); print('Size:',round(candidate.stat().st_size/1024/1024,2),'MB')
files.download(str(candidate))
